In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


# Setup

In [2]:
pip install dagshub mlflow scikit-learn pandas matplotlib seaborn skops --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 695.1 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 2.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 7.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 13.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.4 MB/s eta 0:00:00
   ━━

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret("DAGSHUB_TOKEN")

In [5]:
import dagshub
import mlflow
import mlflow.sklearn

dagshub.auth.add_app_token(token=dagshub_token)
dagshub.init(repo_owner='ngval22', repo_name='fraud-detection-classification', mlflow=True)
mlflow.get_tracking_uri()

Accessing as ngval22

Initialized MLflow to track repo "ngval22/fraud-detection-classification"

Repository ngval22/fraud-detection-classification initialized!

'https://dagshub.com/ngval22/fraud-detection-classification.mlflow'

# Train on numericals only (TEST)

In [6]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# load and join
train_tx = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
df = train_tx.merge(train_id, on='TransactionID', how='left')

# keep plain numeric columns, drop everything else
X = df.select_dtypes(include='number').drop(columns=['isFraud', 'TransactionID'])
y = df['isFraud']

# fill nulls
X = X.fillna(-999)

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Logistic Regression works better with scaled features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

mlflow.set_experiment("fraud_detection_baseline")

with mlflow.start_run(run_name="logistic_regression_test_run"):
    # Log parameters
    params = {
        "class_weight": "balanced",
        "max_iter": 5000,
        "random_state": 42,
        "solver": "lbfgs"
    }
    mlflow.log_params(params)

    # Train
    lr = LogisticRegression(**params)
    lr.fit(X_train_scaled, y_train)

    # Evaluate
    y_pred = lr.predict_proba(X_val_scaled)[:, 1]
    auc = roc_auc_score(y_val, y_pred)

    # Log metric
    mlflow.log_metric("val_auc", auc)

    # Log model
    mlflow.sklearn.log_model(lr, "logistic_regression", serialization_format="skops")

    print(f"Logistic Regression AUC: {auc:.4f}")
    print(f"Run logged to: {mlflow.get_tracking_uri()}")

2026/04/25 15:58:26 INFO mlflow.tracking.fluent: Experiment with name 'fraud_detection_baseline' does not exist. Creating a new experiment.
2026/04/25 16:12:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Logistic Regression AUC: 0.8398
Run logged to: https://dagshub.com/ngval22/fraud-detection-classification.mlflow
🏃 View run logistic_regression_test_run at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/1/runs/b59f544781d2441e83d75f5da21db944
🧪 View experiment at: https://dagshub.com/ngval22/fraud-detection-classification.mlflow/#/experiments/1
